In [11]:
"""Feature Engineering và thiết kế pipeline
Notebook này dùng chung phần triển khai trong src/feature_pipeline.py, để mã notebook và script không lệch nhau. Dữ liệu hiện tại có nhãn nhị phân Diabetes_binary (0/1); đề cương có chỗ mô tả ba lớp Diabetes_012, cần thống nhất trước khi báo cáo.

Quy trình: chia train/test phân tầng 80/20 với seed 42; so sánh 5-fold CV chỉ trên train. Tập test chưa được đánh giá."""

'Feature Engineering và thiết kế pipeline\nNotebook này dùng chung phần triển khai trong src/feature_pipeline.py, để mã notebook và script không lệch nhau. Dữ liệu hiện tại có nhãn nhị phân Diabetes_binary (0/1); đề cương có chỗ mô tả ba lớp Diabetes_012, cần thống nhất trước khi báo cáo.\n\nQuy trình: chia train/test phân tầng 80/20 với seed 42; so sánh 5-fold CV chỉ trên train. Tập test chưa được đánh giá.'

In [12]:
"""1. Đọc dữ liệu và kiểm tra
Chạy các ô từ trên xuống dưới tại thư mục gốc dự án."""

'1. Đọc dữ liệu và kiểm tra\nChạy các ô từ trên xuống dưới tại thư mục gốc dự án.'

In [13]:
from pathlib import Path
import sys, json, os
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.metrics import make_scorer, f1_score

# Tìm root từ thư mục kernel, các thư mục cha, rồi các thư mục con gần đó.
# Hỗ trợ cả src/feature_pipeline.py lẫn src/feature_pipeline/feature_pipeline.py.
def _has_project_files(folder):
    script_options = [
        folder / 'src' / 'feature_pipeline.py',
        folder / 'src' / 'feature_pipeline' / 'feature_pipeline.py',
        folder / 'src' / 'feature_pipeline' / 'feature_pipline.py',
    ]
    data_files = [folder / 'DATA' / 'raw' / 'diabetes.csv', folder / 'sources' / 'diabetes.csv']
    return any(f.is_file() for f in data_files) and any(f.is_file() for f in script_options)

_candidates = [Path.cwd(), *Path.cwd().parents]
# Đường dẫn dự án notebook thực tế được cung cấp.
_actual_root = Path(r'C:\Users\Welcome\Desktop\ML cuối kỳ\MAY_HOC_NHOM\Nhom_Du_An_Machine_Learning\Đồ_Án_Cuối_Kỳ')
_candidates.insert(0, _actual_root)
# Thêm đường dẫn dự án đã nêu, tính từ thư mục kernel và thư mục người dùng.
for _base in [Path.cwd(), *Path.cwd().parents, Path.home()]:
    _named = _base / 'MAY_HOC_NHOM' / 'Đồ_án_cuối_kỳ'
    if _named not in _candidates:
        _candidates.append(_named)
ROOT = next((folder for folder in _candidates if _has_project_files(folder)), None)
if ROOT is None:
    # Notebook có thể được mở từ thư mục con của dự án.
    for base in [Path.cwd(), *Path.cwd().parents[:2]]:
        for current, dirs, files in os.walk(base):
            depth = len(Path(current).relative_to(base).parts)
            dirs[:] = [d for d in dirs if d not in {'.git', '.venv', 'venv', '__pycache__'}]
            if depth > 4:
                dirs[:] = []
                continue
            folder = Path(current)
            if _has_project_files(folder):
                ROOT = folder
                break
        if ROOT is not None:
            break
if ROOT is None:
    raise FileNotFoundError(
        'Không tìm thấy dự án MAY_HOC_NHOM/Đồ_án_cuối_kỳ. Kiểm tra trong thư mục dự án có DATA/raw/diabetes.csv hoặc sources/diabetes.csv, cùng một trong hai tệp '
        'src/feature_pipeline.py hoặc src/feature_pipeline/feature_pipeline.py. '
        'Hãy mở notebook trong dự án hoặc đặt ROOT = Path(r"đường_dẫn_đến_thư_mục_dự_án") thủ công.'
    )

_SCRIPT_CANDIDATES = [
    ROOT / 'src' / 'feature_pipeline.py',
    ROOT / 'src' / 'feature_pipeline' / 'feature_pipeline.py',
    ROOT / 'src' / 'feature_pipeline' / 'feature_pipline.py',
]
_SCRIPT = next((path for path in _SCRIPT_CANDIDATES if path.is_file()), None)
if _SCRIPT is None:
    raise FileNotFoundError(f'Không tìm thấy file mã nguồn trong {ROOT / "src"}')

# Nạp trực tiếp theo đường dẫn để hỗ trợ cả tên feature_pipline.py bị gõ thiếu chữ.
import importlib.util
_spec = importlib.util.spec_from_file_location('feature_pipeline_impl', _SCRIPT)
_impl = importlib.util.module_from_spec(_spec)
sys.modules[_spec.name] = _impl
_spec.loader.exec_module(_impl)
HealthFeatureEngineer = _impl.HealthFeatureEngineer
make_pipeline = _impl.make_pipeline
TARGET = _impl.TARGET
RANDOM_STATE = _impl.RANDOM_STATE

DATA_PATH = next((path for path in [ROOT / 'DATA' / 'raw' / 'diabetes.csv', ROOT / 'sources' / 'diabetes.csv'] if path.is_file()), None)
if DATA_PATH is None:
    raise FileNotFoundError('Không tìm thấy diabetes.csv trong DATA/raw hoặc sources.')
df = pd.read_csv(DATA_PATH)
print(f'Đã tìm thấy dự án tại: {ROOT}')
print(f'Đã đọc dữ liệu từ: {DATA_PATH.relative_to(ROOT)}')
print(f'Kích thước: {df.shape[0]:,} dòng × {df.shape[1]} cột')
print(f'Số ô thiếu: {df.isna().sum().sum():,}')
print(f'Dòng trùng: {df.duplicated().sum():,} ({df.duplicated().mean():.2%})')
print('Phân bố nhãn:')
print(df[TARGET].value_counts().sort_index())
df.head()

Đã tìm thấy dự án tại: C:\Users\Welcome\Desktop\ML cuối kỳ\MAY_HOC_NHOM\Nhom_Du_An_Machine_Learning\Đồ_Án_Cuối_Kỳ
Đã đọc dữ liệu từ: DATA\raw\diabetes.csv
Kích thước: 253,680 dòng × 22 cột
Số ô thiếu: 0
Dòng trùng: 24,206 (9.54%)
Phân bố nhãn:
Diabetes_binary
0    218334
1     35346
Name: count, dtype: int64


,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,Veggies,...,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income,Diabetes_binary
0,1,1,1,40,1,0,0,0,0,1,...,0,5,18,15,1,0,9,4,3,0
1,0,0,0,25,1,0,0,1,0,0,...,1,3,0,0,0,0,7,6,1,0
2,1,1,1,28,0,0,0,0,1,0,...,1,5,30,30,1,0,9,4,8,0
3,1,0,1,27,0,0,0,1,1,1,...,0,2,0,0,0,0,11,3,6,0
4,1,1,1,24,0,0,0,1,1,1,...,0,2,3,0,0,0,11,5,4,0


In [14]:
"""2. Chia dữ liệu và xem đặc trưng mới
Dòng trùng được giữ lại như đã ghi trong README và Experiment Log. Các phép imputation và chuẩn hóa được học bên trong từng fold của pipeline."""

'2. Chia dữ liệu và xem đặc trưng mới\nDòng trùng được giữ lại như đã ghi trong README và Experiment Log. Các phép imputation và chuẩn hóa được học bên trong từng fold của pipeline.'

In [15]:
X = df.drop(columns=TARGET)
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
print(f'Train: {len(X_train):,}; test giữ riêng: {len(X_test):,}')

engineered_preview = HealthFeatureEngineer().fit_transform(X_train.head())
engineered_preview

Train: 202,944; test giữ riêng: 50,736


,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,Veggies,...,PhysHlth,DiffWalk,Sex,Age,Education,Income,BMI_WHO_Group,Comorbidity_Count,Healthy_Lifestyle_Score,Age_x_HighBP
153147,0,0,1,28,1,0,0,1,1,1,...,0,0,1,2,4,5,2.0,0,3,0
176137,1,0,1,23,1,0,0,1,1,1,...,0,0,1,13,4,7,1.0,1,3,13
175578,1,1,1,29,0,0,0,1,1,1,...,0,0,1,9,6,8,2.0,2,3,9
177887,1,1,1,39,0,0,0,0,0,0,...,0,0,1,7,4,7,3.0,2,0,7
182143,0,1,1,16,1,0,0,1,1,1,...,30,1,0,7,5,1,0.0,1,3,0


In [16]:
"""Các đặc trưng mới gồm nhóm BMI theo ngưỡng WHO (<18,5; 18,5–<25; 25–<30; ≥30), tổng bốn bệnh đồng mắc, điểm lối sống 0–3 và tương tác Age × HighBP."""

'Các đặc trưng mới gồm nhóm BMI theo ngưỡng WHO (<18,5; 18,5–<25; 25–<30; ≥30), tổng bốn bệnh đồng mắc, điểm lối sống 0–3 và tương tác Age × HighBP.'

In [17]:
"""3. So sánh cùng mô hình trên cùng các fold
Macro F1 là độ đo chính; Accuracy được ghi nhận để tham khảo. Cả hai nhánh dùng Logistic Regression với class_weight=balanced."""

'3. So sánh cùng mô hình trên cùng các fold\nMacro F1 là độ đo chính; Accuracy được ghi nhận để tham khảo. Cả hai nhánh dùng Logistic Regression với class_weight=balanced.'

In [18]:
scoring = {
    'macro_f1': make_scorer(f1_score, average='macro'),
    'accuracy': 'accuracy',
}
# Chạy tuần tự để tránh joblib tạo tiến trình con cần import module tùy biến.
results = {}
for use_engineered, label in [(False, 'Không thêm đặc trưng'), (True, 'Có đặc trưng mới')]:
    scores = cross_validate(
        make_pipeline(use_engineered), X_train, y_train, cv=cv,
        scoring=scoring, n_jobs=1, return_train_score=False
    )
    results[label] = {
        'Macro F1 trung bình': scores['test_macro_f1'].mean(),
        'Macro F1 độ lệch chuẩn': scores['test_macro_f1'].std(),
        'Accuracy trung bình': scores['test_accuracy'].mean(),
    }
results_df = pd.DataFrame(results).T
results_df.round(6)

,Macro F1 trung bình,Macro F1 độ lệch chuẩn,Accuracy trung bình
Không thêm đặc trưng,0.633365,0.001411,0.731951
Có đặc trưng mới,0.632064,0.001505,0.728994


In [19]:
"""4. Ghi kết quả vào Experiment Log
Ô dưới đây ghi kết quả của lần chạy notebook vào CSV và JSON. Tập test vẫn chưa được dùng."""

'4. Ghi kết quả vào Experiment Log\nÔ dưới đây ghi kết quả của lần chạy notebook vào CSV và JSON. Tập test vẫn chưa được dùng.'

In [20]:
from datetime import date
import platform
import sklearn

log_dir = ROOT / 'logs' / 'Log_02_Train'
log_dir.mkdir(parents=True, exist_ok=True)
csv_log_path = log_dir / 'log_Train.csv'
json_log_path = log_dir / 'log_Train.json'
rows = []
for label, scores in results.items():
    rows.append({
        'date': date.today().isoformat(),
        'runner': 'notebooks/feature_pipeline.ipynb',
        'model': 'LogisticRegression',
        'parameters': 'class_weight=balanced, solver=liblinear, max_iter=500',
        'data_split': 'Stratified 80/20, seed=42; CV chỉ trên train; test chưa đánh giá',
        'metric': f"5-fold Macro F1={scores['Macro F1 trung bình']:.6f} ± {scores['Macro F1 độ lệch chuẩn']:.6f}; accuracy={scores['Accuracy trung bình']:.6f}",
        'remarks': label,
    })
pd.DataFrame(rows).to_csv(csv_log_path, index=False, encoding='utf-8-sig')
summary = json.loads(json_log_path.read_text(encoding='utf-8-sig')) if json_log_path.exists() else {}
summary['notebook_run_date'] = date.today().isoformat()
summary['notebook_results'] = {k: {name: float(value) for name, value in row.items()} for k, row in results.items()}
summary['notebook_status'] = 'CV complete; holdout test not evaluated'
json_log_path.write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding='utf-8')
print(f'Đã cập nhật: {csv_log_path}')
print(f'Đã cập nhật: {json_log_path}')

Đã cập nhật: C:\Users\Welcome\Desktop\ML cuối kỳ\MAY_HOC_NHOM\Nhom_Du_An_Machine_Learning\Đồ_Án_Cuối_Kỳ\logs\Log_02_Train\log_Train.csv
Đã cập nhật: C:\Users\Welcome\Desktop\ML cuối kỳ\MAY_HOC_NHOM\Nhom_Du_An_Machine_Learning\Đồ_Án_Cuối_Kỳ\logs\Log_02_Train\log_Train.json


In [21]:
"""Ghi chú
Không xem điểm số trên test trong giai đoạn chọn đặc trưng/mô hình. Sau khi lựa chọn hoàn tất, nhóm mới đánh giá test một lần và lưu riêng kết quả cuối. Sơ đồ quy trình nằm tại figures/pipeline.png."""

'Ghi chú\nKhông xem điểm số trên test trong giai đoạn chọn đặc trưng/mô hình. Sau khi lựa chọn hoàn tất, nhóm mới đánh giá test một lần và lưu riêng kết quả cuối. Sơ đồ quy trình nằm tại figures/pipeline.png.'